### Try-It Activity 18.1: Comparing Methods


This Try-It activity focuses on weighing the positives and negatives of different estimators and vectorization strategies for a text classification problem.  In order to consider each of these components, you should make use of the `Pipeline` and `GridSearchCV` objects in Scikit-Learn to try different combinations of vectorizers with different estimators.  For each of these, you also want to use the `.cv_results_` to examine the time for the estimator to fit the data.

### The Data

The dataset below is from [kaggle]() and contains a dataset named the "ColBert Dataset" created for this [paper](https://arxiv.org/pdf/2004.12765.pdf).  You are to use the text column to classify whether or not the text was humorous.  It is loaded and displayed below.


In [ ]:

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
)
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier


In [ ]:
df = pd.read_csv('data/dataset-minimal.csv')

In [ ]:
df.head()

#### Task


**Text preprocessing:** As a pre-processing step, perform both `stemming` and `lemmatizing` to normalize your text before classifying. For each technique use both the `CountVectorize`r and `TfidifVectorizer` and use options for stop words and max features to prepare the text data for your estimator.

**Classification:** Once you have prepared the text data with stemming lemmatizing techniques, consider `LogisticRegression`, `DecisionTreeClassifier`, and `MultinomialNB` as classification algorithms for the data. Compare their performance in terms of accuracy and speed.

Share the results of your best classifier in the form of a table with the best version of each estimator, a dictionary of the best parameters and the best score.

In [ ]:
X = df['text']
y = df["humor"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
params = {
    "tree__criterion": ["gini"],
    "tree__max_depth": [3, 4, 5],
    "tree__min_samples_split": [4, 5],
    "tree__min_samples_leaf": [3, 4, 5],
}
tree_pipe = Pipeline(
    [
        (
            "cvect",
            CountVectorizer(stop_words='english', max_features=1500, max_df=0.75),
        ),

        ("tree", DecisionTreeClassifier(random_state=42)),
    ]
)
grid_tree = GridSearchCV(tree_pipe, param_grid=params, verbose=1)
grid_tree.fit(X_train, y_train)

In [ ]:

params = {
    "tree__criterion": ["gini"],
    "tree__max_depth": [3, 4, 5],
    "tree__min_samples_split": [4, 5],
    "tree__min_samples_leaf": [3, 4, 5],
}
tree_pipe2 = Pipeline(
    [
        (
            "tfidf",
            TfidfVectorizer(stop_words='english', max_features=1500, max_df=0.75),
        ),
        ("tree", DecisionTreeClassifier(random_state=42)),
    ]
)
grid_tree2 = GridSearchCV(tree_pipe2, param_grid=params, verbose=1)
grid_tree2.fit(X_train, y_train)

In [ ]:
params = {
    "lr__max_iter": [500, 1000],
    "lr__C": [0.01, 0.1, 1],
}
lr_pipe = Pipeline(
    [
        (
            "cvect",
            CountVectorizer(stop_words='english', max_features=1500, max_df=0.75),
        ),
        ("lr", LogisticRegression(random_state=42)),
    ]
)
grid_lr = GridSearchCV(lr_pipe, param_grid=params, verbose=1)
grid_lr.fit(X_train, y_train)

In [ ]:
params = {
    "lr__max_iter": [500, 1000],
    "lr__C": [0.01, 0.1, 1],
}
lr_pipe2 = Pipeline(
    [

        (
            "tfidf",
            TfidfVectorizer(stop_words='english', max_features=1500, max_df=0.75),
        ),
        ("lr", LogisticRegression(random_state=42)),
    ]
)
grid_lr2 = GridSearchCV(lr_pipe2, param_grid=params, verbose=1)
grid_lr2.fit(X_train, y_train)

In [ ]:
params = {
    "bn__alpha": [0.1, 1, 10, 100],
}
bn_pipe = Pipeline(
    [
        (
            "cvect",
            CountVectorizer(stop_words='english', max_features=1500, max_df=0.75),
        ),
        ("bn", MultinomialNB()),
    ]
)
grid_bn = GridSearchCV(bn_pipe, param_grid=params, verbose=1)
grid_bn.fit(X_train, y_train)

In [ ]:
params = {
    "bn__alpha": [0.1, 1, 10, 100],
}
bn_pipe2 = Pipeline(
    [
        (
            "tfidf",
            TfidfVectorizer(stop_words='english', max_features=1500, max_df=0.75),
        ),
        ("bn", MultinomialNB()),
    ]
)
grid_bn2 = GridSearchCV(bn_pipe2, param_grid=params, verbose=1)
grid_bn2.fit(X_train, y_train)

In [ ]:
pd.DataFrame({'model': ['Logistic CVect', 'Logistic TFIDF', 'Decision Tree Cvect', 'Decision Tree TFIDF', 'Bayes Cvect',
                        'Bayes TFIDF'],
              'best_params': [
                  grid_lr.best_params_,
                  grid_lr2.best_params_,
                  grid_tree.best_params_,
                  grid_tree2.best_params_,
                  grid_bn.best_params_,
                  grid_bn2.best_params_
              ],
              'best_score': [
                  grid_lr.score(X_test, y_test),
                  grid_lr2.score(X_test, y_test),
                  grid_tree.score(X_test, y_test),
                  grid_tree2.score(X_test, y_test),
                  grid_bn.score(X_test, y_test),
                  grid_bn2.score(X_test, y_test)
              ]}).set_index('model')